<img src="https://raw.githubusercontent.com/alan-barzilay/NLPortugues/master/imagens/logo_nlportugues.png"   width="150" align="right">


# Lista 6 - LSTM & GRU 


______________



O objetivo desta lista é fazer com que vocês treinem um modelo de análise de sentimentos utilizando GRU's e LSTM's. Essa lista é semelhante a lista 03 onde aprendemos a usar embeddings e onde você ja recebeu a arquitetura do seu modelo quase pronta. A diferença é que desta vez você ira construir sozinho sua rede e utilizará as camadas que acabamos de aprender: LSTM e GRU.

In [1]:
import tensorflow as tf
from tensorflow import keras
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

I0000 00:00:1779305082.928634  405289 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1779305083.004378  405289 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1779305085.783497  405289 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.


In [2]:
tf.__version__

'2.21.0'

## Importando os dados como um dataframe

Para esta lista nós utilizaremos um recorte do dataset **B2W-Reviews01** que consiste em avaliações de mais de 130k compras online no site Americanas.com e [esta disponivel no github](https://github.com/b2wdigital/b2w-reviews01) sob a licensa CC BY-NC-SA 4.01.

In [3]:
!mkdir data

In [4]:
!curl https://raw.githubusercontent.com/alan-barzilay/NLPortugues/master/Semana%2003/data/b2w-10k.csv --output 'data/b2w-10k.csv'

  % Total    % Received % Xferd  Average Speed  Time    Time    Time   Current
                                 Dload  Upload  Total   Spent   Left   Speed
100  3.82M 100  3.82M   0      0  2.79M      0   00:01   00:01          2.09M


In [5]:
b2wCorpus = pd.read_csv("data/b2w-10k.csv")
b2wCorpus.head()

,submission_date,reviewer_id,product_id,product_name,product_brand,site_category_lv1,site_category_lv2,review_title,overall_rating,recommend_to_a_friend,review_text,reviewer_birth_year,reviewer_gender,reviewer_state,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18
0,2018-01-01 00:11:28,d0fb1ca69422530334178f5c8624aa7a99da47907c44de...,132532965,Notebook Asus Vivobook Max X541NA-GO472T Intel...,NaN,Informática,Notebook,Bom,4,Yes,Estou contente com a compra entrega rápida o ú...,1958,F,RJ,NaN,NaN,NaN,NaN,NaN
1,2018-01-01 00:13:48,014d6dc5a10aed1ff1e6f349fb2b059a2d3de511c7538a...,22562178,Copo Acrílico Com Canudo 500ml Rocie,NaN,Utilidades Domésticas,"Copos, Taças e Canecas","Preço imbatível, ótima qualidade",4,Yes,"Por apenas R$1994.20,eu consegui comprar esse ...",1996,M,SC,NaN,NaN,NaN,NaN,NaN
2,2018-01-01 00:26:02,44f2c8edd93471926fff601274b8b2b5c4824e386ae4f2...,113022329,Panela de Pressão Elétrica Philips Walita Dail...,philips walita,Eletroportáteis,Panela Elétrica,ATENDE TODAS AS EXPECTATIVA.,4,Yes,SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...,1984,M,SP,NaN,NaN,NaN,NaN,NaN
3,2018-01-01 00:35:54,ce741665c1764ab2d77539e18d0e4f66dde6213c9f0863...,113851581,Betoneira Columbus - Roma Brinquedos,roma jensen,Brinquedos,Veículos de Brinquedo,presente mais que desejado,4,Yes,MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...,1985,F,SP,NaN,NaN,NaN,NaN,NaN
4,2018-01-01 01:00:28,7d7b6b18dda804a897359276cef0ca252f9932bf4b5c8e...,131788803,"Smart TV LED 43"" LG 43UJ6525 Ultra HD 4K com C...",lg,TV e Home Theater,TV,"Sem duvidas, excelente",5,Yes,"A entrega foi no prazo, as americanas estão de...",1994,M,MG,NaN,NaN,NaN,NaN,NaN


In [6]:
b2wCorpus["review_text"]

0       Estou contente com a compra entrega rápida o ú...
1       Por apenas R$1994.20,eu consegui comprar esse ...
2       SUPERA EM AGILIDADE E PRATICIDADE OUTRAS PANEL...
3       MEU FILHO AMOU! PARECE DE VERDADE COM TANTOS D...
4       A entrega foi no prazo, as americanas estão de...
                              ...                        
9994    Celular muito rápido, com processador e armaze...
9995    achei o produto muito frágil, o material veio ...
9996    Uma porcaria pois ñ recebi ñ recomendo pra nin...
9997    Maquina excelente,super pratica. recomendo.ent...
9998    Agradeço pelo compromisso, obrigado. ,...........
Name: review_text, Length: 9999, dtype: str


## Pré-processamento 
# <font color='blue'>Questão 1 </font>
Copie suas etapas de préprocessamento da lista 03, ou seja, selecione apenas as colunas relevantes ("review_text" e "recommend_to_a_friend"), converta a coluna "review_text" de uma coluna de `str` para uma coluna de `int` e separe os dados em teste e treino.


In [12]:
from sklearn.model_selection import train_test_split

In [14]:
# Seu código aqui
data = b2wCorpus.dropna(subset=["review_text", "recommend_to_a_friend"])[["review_text", "recommend_to_a_friend"]].copy()

mask = data["recommend_to_a_friend"].isin(["Yes", "No"])
data = data.loc[mask].copy()
data["recommend_to_a_friend"] = data["recommend_to_a_friend"].map({"Yes": 1, "No": 0}).astype(int)

x = data["review_text"].astype(str).values
y = data["recommend_to_a_friend"].values

x_train, x_val, y_train, y_val = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

## Tokenizando




# <font color='blue'>Questão 2 </font>
Utilizando a camada [`TextVectorization`](https://www.tensorflow.org/api_docs/python/tf/keras/layers/experimental/preprocessing/TextVectorization) tokenize os inputs.
Declare a camada e então chame a função `adapt()` no seu conjunto de treino para adequar o seu vocabulário aos reviews. 

Note que o uso de padding não é mais necessario.

In [15]:
from tensorflow.keras.layers import TextVectorization
from tensorflow.keras import layers

In [17]:
# Seu código aqui
maxlen = 61  
vocab_size = 20000  

vectorizer = TextVectorization(
    max_tokens=vocab_size,
    output_mode="int",  
    output_sequence_length=maxlen, 
)

vectorizer.adapt(x_train)

x_train_vec = vectorizer(x_train)
x_val_vec = vectorizer(x_val)

## LSTM&GRU

Agora vamos juntar a camada do tokenizador a nossa camada [Embedding](https://www.tensorflow.org/api_docs/python/tf/keras/layers/Embedding) e definir o resto de nosso modelo.

#  <font color='blue'>Questão 3 </font>

a) Defina, compile, treine e avalie seu modelo, utilize camadas  [LSTM](https://keras.io/api/layers/recurrent_layers/lstm/).
Atenção a dimensão do input da camada de embedding, lembre se que < OOV > e < PAD > possuem seus próprios tokens.
 
 
 
b) Como foi a performance desta rede em comparação a da lista 3?




**<font color='red'> Sua resposta aqui </font>**

b) A LSTM aprendeu mais lentamente , atingiu menor acurácia de generalização e também apresentou overfitting (diferença de ~12% entre treino e validação na época 5). A única vantagem da LSTM foi um loss de teste ligeiramente menor.

In [18]:
# Seu código aqui
model = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- LSTM ---
    layers.LSTM(128, return_sequences=True),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])

In [20]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])
model.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
model.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 22s 79ms/step - accuracy: 0.8405 - loss: 0.3840 - val_accuracy: 0.8780 - val_loss: 0.3212
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 72ms/step - accuracy: 0.9305 - loss: 0.1997 - val_accuracy: 0.8745 - val_loss: 0.3145
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 19s 74ms/step - accuracy: 0.9662 - loss: 0.1124 - val_accuracy: 0.8580 - val_loss: 0.4635
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 70ms/step - accuracy: 0.9804 - loss: 0.0690 - val_accuracy: 0.8615 - val_loss: 0.4595
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 20s 78ms/step - accuracy: 0.9876 - loss: 0.0449 - val_accuracy: 0.8630 - val_loss: 0.5639
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 18ms/step - accuracy: 0.8630 - loss: 0.5639


[0.5639449954032898, 0.8629999756813049]

#  <font color='blue'>Questão 4 </font>

a) Defina, compile, treine e avalie seu modelo, utilize camadas [GRU](https://keras.io/api/layers/recurrent_layers/gru/).
Atenção a dimensão do input da camada de embedding, lembre se que < OOV > e < PAD > possuem seus próprios tokens.
 
 
 
b) Como foi a performance desta rede em comparação a da lista 3?


**<font color='red'> Sua resposta aqui </font>**

b) A GRU foi mais rápida que a LSTM, mas ainda mais lenta que a rede simples. Assim como a LSTM, a GRU apresentou overfitting e menor capacidade de generalização.

In [21]:
# Seu código aqui
model = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- GRU ---
    layers.GRU(128, return_sequences=True),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])

In [22]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])
model.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
model.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 21s 71ms/step - accuracy: 0.8357 - loss: 0.3980 - val_accuracy: 0.8835 - val_loss: 0.2906
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 69ms/step - accuracy: 0.9347 - loss: 0.1951 - val_accuracy: 0.8830 - val_loss: 0.3216
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 67ms/step - accuracy: 0.9657 - loss: 0.1098 - val_accuracy: 0.8735 - val_loss: 0.3301
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 17s 68ms/step - accuracy: 0.9799 - loss: 0.0636 - val_accuracy: 0.8500 - val_loss: 0.5935
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 18s 70ms/step - accuracy: 0.9852 - loss: 0.0474 - val_accuracy: 0.8570 - val_loss: 0.5732
63/63 ━━━━━━━━━━━━━━━━━━━━ 1s 10ms/step - accuracy: 0.8570 - loss: 0.5732


[0.5732036828994751, 0.8569999933242798]

## Redes Bi-direcionais
#  <font color='blue'>Questão 5 </font>

a) Defina, compile, treine e avalie um novo modelo que utilize contexto em ambas as direções usando a camada [`Bidirectional()`](https://keras.io/api/layers/recurrent_layers/bidirectional/), seja com camadas GRU ou LSTM.


b) Como foi sua performance em relação as questões anteriores com contexto unidirecional?

**<font color='red'> Sua resposta aqui </font>**

b) A Bidirectional não trouxe vantagem sobre o contexto unidirecional. Ao contrário: consumiu mais recursos, sofreu overfitting mais severo e entregou resultados equivalentes (ou piores, considerando o loss) às redes unidirecionais.

In [23]:
# Seu código aqui
model = tf.keras.Sequential([    

    ############ Seu código aqui##################
    # --- EMBEDDING ---
    tf.keras.layers.Embedding(
        input_dim=vocab_size,
        output_dim=128,
    ),

    # --- Bidirectional ---
    layers.Bidirectional(layers.LSTM(128, return_sequences=True)),

    ##############################################
    # Conv1D + global max pooling
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.Conv1D(128, 7, padding='valid', activation='relu', strides=3),
    layers.GlobalMaxPooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(1, activation='sigmoid'),
])

In [24]:
model.compile(optimizer="adam", loss="binary_crossentropy", metrics=['accuracy'])
model.fit(x_train_vec, y_train, epochs=5, validation_data=(x_val_vec, y_val))
model.evaluate(x=x_val_vec, y=y_val)

Epoch 1/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 31s 107ms/step - accuracy: 0.8410 - loss: 0.3893 - val_accuracy: 0.8855 - val_loss: 0.2850
Epoch 2/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 26s 105ms/step - accuracy: 0.9362 - loss: 0.1920 - val_accuracy: 0.8835 - val_loss: 0.2890
Epoch 3/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 27s 108ms/step - accuracy: 0.9665 - loss: 0.1086 - val_accuracy: 0.8740 - val_loss: 0.3773
Epoch 4/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 28s 112ms/step - accuracy: 0.9822 - loss: 0.0616 - val_accuracy: 0.8660 - val_loss: 0.6281
Epoch 5/5
250/250 ━━━━━━━━━━━━━━━━━━━━ 28s 111ms/step - accuracy: 0.9874 - loss: 0.0378 - val_accuracy: 0.8625 - val_loss: 0.9754
63/63 ━━━━━━━━━━━━━━━━━━━━ 2s 25ms/step - accuracy: 0.8625 - loss: 0.9754


[0.9754360318183899, 0.862500011920929]